# 📈 Lecture 8: Scaling Reads — Read Replicas & Connection Pooling

The Spring Sale is live and traffic on the DataCart storefront has spiked. Two very
different pressures show up under load:

1. **Read volume** — thousands of shoppers browsing products, ratings, and promotions
   generate far more *reads* than writes.
2. **Connection count** — a busy web/API tier opens many short-lived database
   connections, which can exhaust Postgres' connection budget.

Lakebase addresses these with two independent features:

| Pressure | Feature | What it does |
|----------|---------|--------------|
| Too many reads | **Read replicas** | Add read-only computes that serve reads from the *same* storage — no data copy |
| Too many connections | **Connection pooling** | A built-in **PgBouncer** pooler multiplexes many client connections onto few server connections |

> 📖 **Docs**: [Read replicas](https://docs.databricks.com/aws/en/oltp/projects/read-replicas) · [Connection pooling](https://docs.databricks.com/aws/en/oltp/projects/connection-pooling)

## Part 1 — Read Replicas

A **read replica** is an independent, **read-only** compute endpoint attached to a branch.
Because Lakebase separates compute from storage, replicas do **not** copy your data —
every compute (the primary read-write endpoint and all replicas) reads from the **same
shared storage layer**, so they see a consistent view of the data.

```
                Branch: production
  ┌────────────────────┬────────────────────┬────────────────────┐
  │  primary            │  read-replica-1     │  read-replica-2     │
  │  (read-write)       │  (read-only)        │  (read-only)        │
  └─────────┬───────────┴──────────┬─────────┴──────────┬─────────┘
            │                      │                    │
            ▼                      ▼                    ▼
  ┌───────────────────────────────────────────────────────────────┐
  │              Shared object storage (no duplication)           │
  └───────────────────────────────────────────────────────────────┘
```

**Key facts (from the docs):**
- You can add **up to 6 read replicas per branch**.
- Replicas involve **no data duplication or replication** — all computes read from the
  same storage, ensuring a consistent source.
- There is **no automatic read/write split**. Your application must connect to the
  **replica endpoint explicitly** to send read traffic to it; writes continue to go to
  the primary read-write endpoint.

### Add a read replica — via the UI

In the Lakebase app, open your project → **production** branch → the **Computes** tab →
click **Add Read Replica** to instantly provision a new read-only compute.

### Add a read replica — via the SDK

A read replica is simply an endpoint with `endpoint_type = ENDPOINT_TYPE_READ_ONLY`.
The snippet below adds one to the `production` branch of the workshop project.

In [0]:
%pip install "databricks-sdk>=0.89.0" -q
%pip install "psycopg[binary]" -q

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Endpoint, EndpointSpec, EndpointType

w = WorkspaceClient()

# Derive the same project name used throughout the workshop
db_user = w.current_user.me().user_name
username_prefix = db_user.split("@")[0].replace(".", "-")
project_name = f"lakebase-branching-workshop-{username_prefix}"
branch_path = f"projects/{project_name}/branches/production"

# Create a read-only endpoint (a read replica) on the production branch
replica = w.postgres.create_endpoint(
    parent=branch_path,
    endpoint_id="read-replica-1",
    endpoint=Endpoint(spec=EndpointSpec(
        endpoint_type=EndpointType.ENDPOINT_TYPE_READ_ONLY,
        autoscaling_limit_min_cu=0.5,
        autoscaling_limit_max_cu=2.0,
    )),
).wait()

print(f"✅ Read replica created on {branch_path}")
print(f"   Endpoint: {replica.name}")

### Connecting to the replica

Point read-heavy queries at the **replica's** host (the primary keeps serving writes).
The connection pattern is identical to the primary — generate an OAuth token for the
replica endpoint and connect with `psycopg`.

In [0]:
import psycopg

replica_host = replica.status.hosts.host
cred = w.postgres.generate_database_credential(endpoint=replica.name)

read_conn = psycopg.connect(
    host=replica_host,
    port=5432,
    dbname="databricks_postgres",
    user=db_user,
    password=cred.token,
    sslmode="require",
)
read_conn.autocommit = True

with read_conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ecommerce.products")
    print(f"📖 Products (read from replica): {cur.fetchone()[0]}")

read_conn.close()

> 💡 **DataCart pattern:** during the Spring Sale, route the storefront's product,
> ratings, and promotions reads to a replica endpoint, leaving the primary read-write
> endpoint free for cart and checkout writes. Because there is no automatic split, this
> is a deliberate choice in the app's connection configuration.

## Part 2 — Connection Pooling (PgBouncer)

Each Postgres server connection consumes memory, so a compute can only hold a limited
number of them. A busy app tier that opens a new connection per request will hit that
ceiling quickly. Lakebase ships with a **built-in PgBouncer pooler** that maintains a
pool of server connections and shares them across many client connections.

**Key facts (from the docs):**
- Supports up to **10,000 concurrent client connections**.
- Runs in **transaction mode** — a server connection is held only for the duration of a
  single transaction, then returned to the pool.
- Pool size is roughly **90% of `max_connections`** (which varies by compute size);
  query timeout is **120 seconds**.
- Requires a **native Postgres password role** — OAuth roles are **not** supported on
  the pooler.
- Enable it from the Lakebase **Connect** dialog by selecting a password role and
  toggling the connection-pooling switch.

### Pooled vs. direct hostnames

The pooler is exposed on a separate hostname. You choose pooled or direct simply by which
host you connect to (port `5432` either way):

| Connection | Hostname pattern |
|------------|------------------|
| Read-write, **pooled** | `<endpoint-id>-pooler.<region>.<cloud>.databricks.com` |
| Read-only, **pooled** | `<endpoint-id>-ro-pooler.<region>.<cloud>.databricks.com` |
| Direct (unpooled) | the standard endpoint hostname (no `-pooler` suffix) |

### When to use the pooler vs. a direct connection

Transaction mode is ideal for short, stateless queries (typical web/API traffic), but it
restricts features that rely on session state. Use a **direct connection** if you need:

- SQL-level **prepared statements** or session-level settings
- **Temporary tables**
- `WITH HOLD` cursors
- **Advisory locks**
- `LISTEN` / `NOTIFY`

> 💡 **DataCart pattern:** the storefront's high-volume, short-lived queries during the
> sale are a perfect fit for the pooled endpoint. Note that the DataCart app authenticates
> with short-lived **OAuth tokens** and pools connections *itself* (via `psycopg_pool`);
> to use the **built-in** PgBouncer pooler instead, connect through the `-pooler` host with
> a **password role**.

## Summary

| Goal | Feature | How |
|------|---------|-----|
| Scale **reads** | Read replicas | Add read-only endpoints (up to 6/branch); connect to the replica host explicitly |
| Scale **connections** | Built-in PgBouncer | Connect via the `-pooler` / `-ro-pooler` host using a password role |

Both features build on Lakebase's compute/storage separation: replicas add read compute
without copying data, and the pooler adds connection capacity without a separate service
to run.

> 📖 **Docs**: [Read replicas](https://docs.databricks.com/aws/en/oltp/projects/read-replicas) · [Connection pooling](https://docs.databricks.com/aws/en/oltp/projects/connection-pooling) · [Connect to your database](https://docs.databricks.com/aws/en/oltp/projects/connect)